# ETL — Preparación y Carga de Datos
## Práctica 1 · SOG2 2S26 · Grupo 8

**Punto 1 — Preparación de datos:**
1. Extraer los datos del archivo `.csv`
2. Verificar si hay valores faltantes o duplicados y decidir cómo manejarlos
3. Asegurarse de que los tipos de datos sean correctos para cada columna
4. Cargar los datos a una base de datos SQL en la nube

**Estrategia de carga:** Incremental (soporta múltiples ejecuciones y múltiples archivos CSV)

---
## 1. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez (o en Google Colab)
# !pip install pandas sqlalchemy psycopg2-binary python-dotenv

## 2. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from datetime import datetime
from sqlalchemy import create_engine, text, inspect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

print(f"pandas: {pd.__version__}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 3. Extracción — Cargar el CSV

El archivo `Venta_online_c.csv` usa **punto y coma (`;`)** como separador, no coma.

In [ ]:
# ============================================================
# Configuración: ruta al CSV
# ============================================================
CSV_PATH = '../Enunciado/Venta_online_c.csv'  # Ajustar según tu entorno

df_raw = pd.read_csv(CSV_PATH, sep=';')

print(f"Archivo cargado: {CSV_PATH}")
print(f"Filas: {df_raw.shape[0]:,}  |  Columnas: {df_raw.shape[1]}")
print(f"\nColumnas: {list(df_raw.columns)}")
df_raw.head(10)

---
## 4. Verificación de calidad de datos

Verificar si hay valores faltantes o duplicados y decidir cómo manejarlos.

In [ ]:
print("="*60)
print("4.1 TIPOS DE DATOS ORIGINALES")
print("="*60)
print(df_raw.dtypes)
print()

In [ ]:
print("="*60)
print("4.2 VALORES NULOS POR COLUMNA")
print("="*60)
nulos = df_raw.isnull().sum()
print(nulos)
print(f"\nTotal de valores nulos en el dataset: {nulos.sum()}")

if nulos.sum() == 0:
    print("\nNo se encontraron valores nulos. No se requiere imputación.")
else:
    print("\nSe encontraron valores nulos. Requiere tratamiento.")

In [ ]:
print("="*60)
print("4.3 FILAS DUPLICADAS")
print("="*60)
duplicadas = df_raw.duplicated().sum()
print(f"Filas completamente duplicadas: {duplicadas}")

dup_id = df_raw['Id_cliente'].duplicated().sum()
print(f"Id_cliente duplicados: {dup_id}")

if duplicadas == 0 and dup_id == 0:
    print("\nNo hay filas duplicadas ni Id_cliente repetidos.")
else:
    print("\nSe encontraron duplicados. Se procederá a eliminarlos.")
    df_raw = df_raw.drop_duplicates()
    df_raw = df_raw.drop_duplicates(subset=['Id_cliente'], keep='first')
    print(f"   Filas después de limpiar: {len(df_raw):,}")

In [ ]:
print("="*60)
print("4.4 ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
df_raw.describe()

In [ ]:
print("="*60)
print("4.5 VALORES ÚNICOS EN CAMPOS CATEGÓRICOS")
print("="*60)

categoricos = ['Genero', 'MetodoPago', 'Navegador', 'Boletin', 'Vale']

for col in categoricos:
    valores = sorted(df_raw[col].unique())
    print(f"  {col}: {valores}  (n_unicos = {len(valores)})")

# Verificar que los valores estén dentro de los rangos esperados según el PDF
assert set(df_raw['Genero'].unique()).issubset({0, 1}), "Genero tiene valores fuera de {0, 1}"
assert set(df_raw['MetodoPago'].unique()).issubset({0, 1, 2}), "MetodoPago tiene valores fuera de {0, 1, 2}"
assert set(df_raw['Navegador'].unique()).issubset({0, 1, 2, 3, 4}), "Navegador tiene valores fuera de {0..4}"
assert set(df_raw['Boletin'].unique()).issubset({0, 1}), "Boletin tiene valores fuera de {0, 1}"
assert set(df_raw['Vale'].unique()).issubset({0, 1}), "Vale tiene valores fuera de {0, 1}"

print("\nTodos los campos categóricos tienen valores válidos.")

In [ ]:
print("="*60)
print("4.6 VERIFICACIÓN DE RANGOS NUMÉRICOS")
print("="*60)

print(f"  Edad:        min={df_raw['Edad'].min()}, max={df_raw['Edad'].max()}")
print(f"  Venta_total: min={df_raw['Venta_total'].min()}, max={df_raw['Venta_total'].max()}")
print(f"  N_Compras:   min={df_raw['N_Compras'].min()}, max={df_raw['N_Compras'].max()}")
print(f"  MontoCompra: min={df_raw['MontoCompra'].min()}, max={df_raw['MontoCompra'].max()}")
print(f"  Tiempo:      min={df_raw['Tiempo'].min()}, max={df_raw['Tiempo'].max()}")

# Verificar que no hay valores negativos en campos que no deberían
assert (df_raw['Edad'] > 0).all(), "Hay edades <= 0"
assert (df_raw['Venta_total'] >= 0).all(), "Hay ventas totales negativas"
assert (df_raw['N_Compras'] > 0).all(), "Hay número de compras <= 0"
assert (df_raw['MontoCompra'] >= 0).all(), "Hay montos de compra negativos"
assert (df_raw['Tiempo'] > 0).all(), "Hay tiempos <= 0"

print("\nTodos los rangos numéricos son coherentes y sin valores anómalos.")

### Resumen de verificación de calidad

| Verificación | Resultado | Acción |
|---|---|---|
| Valores nulos | 0 nulos | No se requiere imputación |
| Filas duplicadas | 0 duplicadas | No se requiere eliminación |
| Id_cliente duplicados | 0 duplicados | Cada cliente es único |
| Valores categóricos | Dentro de rango | Validados contra el enunciado |
| Valores numéricos | Coherentes | Sin negativos ni anómalos |

---
## 5. Transformación — Corrección de tipos de datos

Asegurarse de que los tipos de datos sean correctos para cada columna.

In [ ]:
# Trabajamos sobre una copia para preservar el original
df = df_raw.copy()

# ============================================================
# 5.1 Convertir FechaCompra de texto a fecha
# Formato en CSV: DD.MM.AA (ej: 02.02.21 = 2 de febrero de 2021)
# ============================================================
df['FechaCompra'] = pd.to_datetime(df['FechaCompra'], format='%d.%m.%y')

print("Conversión de FechaCompra:")
print(f"  Fecha mínima: {df['FechaCompra'].min()}")
print(f"  Fecha máxima: {df['FechaCompra'].max()}")

# Validar que todas las fechas estén en 2021
assert df['FechaCompra'].dt.year.unique().tolist() == [2021], "Hay fechas fuera de 2021"
print("Todas las fechas corresponden al año 2021")

In [ ]:
# ============================================================
# 5.2 Convertir tipos enteros y booleanos
# ============================================================
for col in ['Genero', 'MetodoPago', 'Navegador']:
    df[col] = df[col].astype('int8')

df['Boletin'] = df['Boletin'].astype(bool)
df['Vale'] = df['Vale'].astype(bool)

# ============================================================
# 5.3 Asegurar 4 decimales en campos monetarios
# ============================================================
df['Venta_total'] = df['Venta_total'].round(4)
df['MontoCompra'] = df['MontoCompra'].round(4)

print("Tipos de datos después de la conversión:")
print(df.dtypes)
print("\nTipos de datos corregidos exitosamente.")

In [ ]:
# ============================================================
# 5.4 Renombrar columnas a snake_case (buena práctica para SQL)
# ============================================================
df = df.rename(columns={
    'Id_cliente':  'id_cliente',
    'Edad':        'edad',
    'Genero':      'genero',
    'Venta_total': 'venta_total',
    'N_Compras':   'n_compras',
    'FechaCompra': 'fecha_compra',
    'MontoCompra': 'monto_compra',
    'MetodoPago':  'metodo_pago',
    'Tiempo':      'tiempo',
    'Navegador':   'navegador',
    'Boletin':     'boletin',
    'Vale':        'vale'
})

print("Columnas renombradas:")
print(list(df.columns))
df.head()

---
## 6. Normalización — Separar en tablas relacionales

Se separan los datos en dos tablas:
- **`clientes`**: perfil del cliente con datos agregados del año (id, edad, género, venta total, número de compras)
- **`compras`**: detalle de compra individual (fecha, monto, método de pago, tiempo, navegador, boletín, vale)

In [ ]:
# ============================================================
# Tabla clientes: perfil y métricas agregadas anuales
# ============================================================
df_clientes = df[['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras']].copy()

print(f"Tabla CLIENTES: {df_clientes.shape[0]:,} filas x {df_clientes.shape[1]} columnas")
df_clientes.head(10)

In [ ]:
# ============================================================
# Tabla compras: detalle de cada compra individual
# ============================================================
df_compras = df[['id_cliente', 'fecha_compra', 'monto_compra', 'metodo_pago',
                 'tiempo', 'navegador', 'boletin', 'vale']].copy()

print(f"Tabla COMPRAS: {df_compras.shape[0]:,} filas x {df_compras.shape[1]} columnas")
df_compras.head(10)

In [ ]:
# ============================================================
# Verificación de integridad referencial
# Todos los id_cliente en compras deben existir en clientes
# ============================================================
ids_clientes = set(df_clientes['id_cliente'])
ids_compras = set(df_compras['id_cliente'])

huerfanos = ids_compras - ids_clientes
print(f"Clientes sin referencia en tabla compras (huérfanos): {len(huerfanos)}")

if len(huerfanos) == 0:
    print("Integridad referencial verificada: todos los id_cliente de compras existen en clientes.")
else:
    print(f"IDs huérfanos encontrados: {huerfanos}")

---
## 7. Carga — Conexión a PostgreSQL en la nube (AWS)

**Punto 1.d:** *"Cargar los datos a una base de datos SQL en la nube."*

In [ ]:
# ============================================================
# CONFIGURACIÓN DE CONEXIÓN
# Ajustar estos valores a tu entorno AWS
# ============================================================
DB_USER     = 'postgres'              # Usuario de PostgreSQL
DB_PASSWORD = 'tu_contraseña'         # Contraseña
DB_HOST     = 'tu_ip_o_endpoint_aws'  # IP o endpoint del contenedor en AWS
DB_PORT     = '5432'                  # Puerto (5432 es el default)
DB_NAME     = 'ventas_online'         # Nombre de la base de datos

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# SEGURIDAD: No dejes credenciales hardcodeadas en producción.
# Usa variables de entorno o un archivo .env
# Ejemplo con dotenv:
# from dotenv import load_dotenv
# load_dotenv()
# DATABASE_URL = os.getenv('DATABASE_URL')

print(f"Conectando a: postgresql://***@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
# ============================================================
# Crear engine de SQLAlchemy y probar conexión
# ============================================================
engine = create_engine(DATABASE_URL)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version()"))
        version = result.scalar()
    print(f"Conexión exitosa")
    print(f"   PostgreSQL: {version}")
except Exception as e:
    print(f"Error de conexión: {e}")
    print("\nVerifica:")
    print("  1. Que el contenedor de PostgreSQL esté corriendo en AWS")
    print("  2. Que el host, puerto, usuario y contraseña sean correctos")
    print("  3. Que el security group/firewall permita conexiones en el puerto 5432")

### 7.1 Verificar que las tablas existan

In [ ]:
# Verificar que las tablas existen
inspector = inspect(engine)
tablas = inspector.get_table_names()
print(f"   Tablas en la base de datos: {tablas}")

### 7.2 Carga incremental de datos

Esta función permite:
- Ejecutar el notebook múltiples veces sin duplicar datos
- Cargar múltiples archivos CSV (solo inserta clientes/compras nuevos)

In [ ]:
def cargar_incremental(df_clientes, df_compras, engine):
    """
    Carga solo los clientes y compras que NO existen aún en la BD.
    Permite ejecutar múltiples veces y con múltiples archivos CSV.

    Args:
        df_clientes: DataFrame con columnas de la tabla clientes
        df_compras:  DataFrame con columnas de la tabla compras
        engine:      SQLAlchemy engine conectado a PostgreSQL

    Returns:
        dict con estadísticas de la carga
    """
    # 1. Obtener IDs que ya existen en la BD
    with engine.connect() as conn:
        existentes = pd.read_sql(
            "SELECT id_cliente FROM clientes", conn
        )['id_cliente'].tolist()

    # 2. Filtrar solo los nuevos
    nuevos_clientes = df_clientes[~df_clientes['id_cliente'].isin(existentes)].copy()
    nuevos_ids = set(nuevos_clientes['id_cliente'])
    nuevas_compras = df_compras[df_compras['id_cliente'].isin(nuevos_ids)].copy()

    stats = {
        'total_en_csv': len(df_clientes),
        'ya_existentes': len(existentes),
        'nuevos_clientes': len(nuevos_clientes),
        'nuevas_compras': len(nuevas_compras),
    }

    # 3. Insertar solo los nuevos
    if not nuevos_clientes.empty:
        nuevos_clientes.to_sql('clientes', engine, if_exists='append', index=False)
        nuevas_compras.to_sql('compras', engine, if_exists='append', index=False)
        print(f"Insertados {len(nuevos_clientes):,} clientes nuevos "
              f"y {len(nuevas_compras):,} compras nuevas")
    else:
        print(f"No hay datos nuevos para insertar "
              f"({len(existentes):,} clientes ya existen en la BD)")

    # 4. Verificar totales en la BD
    with engine.connect() as conn:
        total_c = conn.execute(text("SELECT COUNT(*) FROM clientes")).scalar()
        total_p = conn.execute(text("SELECT COUNT(*) FROM compras")).scalar()

    stats['total_clientes_bd'] = total_c
    stats['total_compras_bd'] = total_p
    print(f"Total en BD: {total_c:,} clientes, {total_p:,} compras")

    return stats

In [ ]:
# ============================================================
# Ejecutar carga incremental
# ============================================================
print("Iniciando carga de datos...\n")
stats = cargar_incremental(df_clientes, df_compras, engine)

print("\n" + "="*60)
print("RESUMEN DE CARGA")
print("="*60)
for k, v in stats.items():
    print(f"  {k}: {v:,}")

---
## 8. Verificación final — Consultar la BD

Validamos que los datos cargados en PostgreSQL coincidan con el CSV original.

In [ ]:
print("="*60)
print("8.1 CONTEO DE REGISTROS")
print("="*60)

with engine.connect() as conn:
    n_clientes = conn.execute(text("SELECT COUNT(*) FROM clientes")).scalar()
    n_compras = conn.execute(text("SELECT COUNT(*) FROM compras")).scalar()

print(f"  Clientes en BD:  {n_clientes:,}  (esperado: {len(df_clientes):,})")
print(f"  Compras en BD:   {n_compras:,}  (esperado: {len(df_compras):,})")

assert n_clientes == len(df_clientes), f"Error: clientes BD ({n_clientes}) != CSV ({len(df_clientes)})"
assert n_compras == len(df_compras), f"Error: compras BD ({n_compras}) != CSV ({len(df_compras)})"
print("\nLos conteos coinciden correctamente.")

In [ ]:
print("="*60)
print("8.2 MUESTRA DE DATOS — TABLA CLIENTES")
print("="*60)

with engine.connect() as conn:
    muestra_clientes = pd.read_sql(
        "SELECT * FROM clientes ORDER BY id_cliente LIMIT 10", conn
    )
muestra_clientes

In [ ]:
print("="*60)
print("8.3 MUESTRA DE DATOS — TABLA COMPRAS")
print("="*60)

with engine.connect() as conn:
    muestra_compras = pd.read_sql(
        "SELECT * FROM compras ORDER BY id_compra LIMIT 10", conn
    )
muestra_compras

In [ ]:
print("="*60)
print("8.4 VERIFICACIÓN JOIN — INTEGRIDAD REFERENCIAL EN BD")
print("="*60)

with engine.connect() as conn:
    resultado = pd.read_sql("""
        SELECT
            c.id_cliente, c.edad, c.genero, c.venta_total,
            p.fecha_compra, p.monto_compra, p.metodo_pago, p.navegador
        FROM clientes c
        JOIN compras p ON c.id_cliente = p.id_cliente
        ORDER BY c.id_cliente
        LIMIT 10
    """, conn)

print(f"JOIN exitoso: {len(resultado)} filas")
resultado

In [ ]:
print("="*60)
print("8.5 VERIFICACIÓN RÁPIDA — DATOS MONETARIOS (4 decimales)")
print("="*60)

with engine.connect() as conn:
    monetario = pd.read_sql("""
        SELECT
            c.id_cliente,
            c.venta_total,
            p.monto_compra
        FROM clientes c
        JOIN compras p ON c.id_cliente = p.id_cliente
        ORDER BY c.id_cliente
        LIMIT 5
    """, conn)

print("Verificando que los campos monetarios mantengan precisión:")
print(monetario.to_string(index=False))
print("\nCampos monetarios con precisión NUMERIC(12,4) en la BD.")

---
## 9. (Opcional) Carga de múltiples archivos CSV

Si necesitas cargar más archivos CSV en el futuro, usa esta celda.  
Solo insertará los clientes y compras que **no existan** aún en la BD.

In [ ]:
# ============================================================
# Carga de múltiples archivos CSV
# Descomentar y ajustar la ruta según sea necesario
# ============================================================

# CARPETA_CSV = '../Enunciado/'  # Carpeta donde están los CSVs
# archivos = glob.glob(os.path.join(CARPETA_CSV, '*.csv'))
#
# for archivo in archivos:
#     print(f"\n{'='*60}")
#     print(f"Procesando: {os.path.basename(archivo)}")
#     print(f"{'='*60}")
#
#     # Leer CSV
#     df_temp = pd.read_csv(archivo, sep=';')
#
#     # Transformar
#     df_temp['FechaCompra'] = pd.to_datetime(df_temp['FechaCompra'], format='%d.%m.%y')
#     for col in ['Genero', 'MetodoPago', 'Navegador']:
#         df_temp[col] = df_temp[col].astype('int8')
#     df_temp['Boletin'] = df_temp['Boletin'].astype(bool)
#     df_temp['Vale'] = df_temp['Vale'].astype(bool)
#     df_temp['Venta_total'] = df_temp['Venta_total'].round(4)
#     df_temp['MontoCompra'] = df_temp['MontoCompra'].round(4)
#
#     # Renombrar columnas
#     df_temp = df_temp.rename(columns={
#         'Id_cliente': 'id_cliente', 'Edad': 'edad', 'Genero': 'genero',
#         'Venta_total': 'venta_total', 'N_Compras': 'n_compras',
#         'FechaCompra': 'fecha_compra', 'MontoCompra': 'monto_compra',
#         'MetodoPago': 'metodo_pago', 'Tiempo': 'tiempo',
#         'Navegador': 'navegador', 'Boletin': 'boletin', 'Vale': 'vale'
#     })
#
#     # Separar
#     c = df_temp[['id_cliente', 'edad', 'genero', 'venta_total', 'n_compras']].copy()
#     p = df_temp[['id_cliente', 'fecha_compra', 'monto_compra', 'metodo_pago',
#                  'tiempo', 'navegador', 'boletin', 'vale']].copy()
#
#     # Cargar incremental
#     cargar_incremental(c, p, engine)

print("Descomentar el código de arriba para cargar múltiples CSVs.")

---
## 10. Resumen final del ETL

| Fase | Descripción | Estado |
|---|---|---|
| **Extracción** | CSV cargado con `sep=';'` | ✅ |
| **Verificación** | 0 nulos, 0 duplicados, rangos válidos | ✅ |
| **Transformación** | Tipos corregidos, fechas parseadas, columnas renombradas | ✅ |
| **Normalización** | Separado en tablas `clientes` y `compras` | ✅ |
| **Carga** | Datos insertados en PostgreSQL (AWS) con estrategia incremental | ✅ |
| **Verificación BD** | Conteos, muestras y JOIN verificados | ✅ |

### Características del ETL:
- **Idempotente:** puede ejecutarse múltiples veces sin duplicar datos
- **Incremental:** solo inserta clientes/compras que no existen en la BD
- **Multi-archivo:** preparado para cargar múltiples CSVs (sección 9)
- **Monetarios:** NUMERIC(12,4) con mínimo 4 decimales